# FASTQ to counts
This notebook shows how to use basic tools to transform transcriptomics FASTQ files to generate count files which are typically later analyzed in Python/R.
We will download all the FASTQ files,
Perform QC with FastQC,
Trim? Re-QC
Align the reads to the genome with STARsolo to assess data quality,
and align the reads to the transcriptome to generate the counts file. 

Running this notebook locally requires X GB of space (X for FASTQs, Y for STAR genome, Z for transcriptome),
And ABC GB RAM.


In [ ]:
%%bash
set -euo pipefail

download() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local sdrf_url="https://www.ebi.ac.uk/biostudies/files/${accession}/${accession}.sdrf.txt"
    local sdrf_file="${out_dir}/${accession}.sdrf.txt"
    local urls_file="${out_dir}/read_fastq_urls.txt"
    local fastq_dir="${out_dir}/fastq"

    mkdir -p "${fastq_dir}"

    # Download the SDRF sample sheet.
    curl -L "${sdrf_url}" -o "${sdrf_file}"

    # Extract only biological read FASTQs.
    # For 10x scRNA-seq counting, we want R1 + R2 and do not need I1/I2 here.
    tr '\t' '\n' < "${sdrf_file}" \
        | grep -Eo 'ftp://[^[:space:]]+_(R1|R2)_001\.fastq\.gz' \
        | sort -u > "${urls_file}"

    printf "FASTQ URLs to download: %s\n" "$(wc -l < "${urls_file}")"

    # Download R1/R2 FASTQs in parallel.
    xargs -a "${urls_file}" -n 1 -P 4 wget -c -P "${fastq_dir}"
}

download_if_missing() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local fastq_dir="${out_dir}/fastq"

    # Treat the download as complete only if we already have biological reads.
    if compgen -G "${fastq_dir}/*_R1_001.fastq.gz" > /dev/null && \
       compgen -G "${fastq_dir}/*_R2_001.fastq.gz" > /dev/null; then
        printf "R1/R2 FASTQs already present in %s\n" "${fastq_dir}"
    else
        download "${data_dir}"
    fi
}

download_if_missing "${1:-data}"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 21173  100 21173    0     0  14516      0  0:00:01  0:00:01 --:--:-- 48561
main: line 29: aria2c: command not found


CalledProcessError: Command 'b'set -euo pipefail\n\ndownload() {\n    local data_dir="${1:-data}"\n    local accession="E-MTAB-13632"\n    local out_dir="${data_dir}/${accession}"\n    local sdrf_url="https://www.ebi.ac.uk/biostudies/files/${accession}/${accession}.sdrf.txt"\n    local sdrf_file="${out_dir}/${accession}.sdrf.txt"\n    local urls_file="${out_dir}/fastq_urls.txt"\n    local fastq_dir="${out_dir}/fastq"\n\n    mkdir -p "${fastq_dir}"\n\n    # Download SDRF\n    curl -L "${sdrf_url}" -o "${sdrf_file}"\n\n    # Extract every FTP FASTQ link from the SDRF, regardless of the exact column name.\n    tr \'\\t\' \'\\n\' < "${sdrf_file}" \\\n        | grep -Eo \'ftp://[^[:space:]]+\\.fastq\\.gz\' \\\n        | sort -u > "${urls_file}"\n\n    # Download all FASTQs\n    # wget \\\n    #     --continue \\\n    #     --directory-prefix="${fastq_dir}" \\\n    #     --input-file="${urls_file}"\n\n\taria2c \\\n\t\t--continue=true \\\n\t\t--dir="${fastq_dir}" \\\n\t\t--input-file="${urls_

In [2]:
%%bash
set -euo pipefail

uname -a
echo "SHELL=$SHELL"
which bash
which python
python --version
fastqc --version
samtools --version | head -n 1
STAR --version
pwd

Linux DekLeg 6.6.87.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux
SHELL=/bin/bash
/usr/bin/bash
/home/dekel/miniforge3/envs/scrna-prep/bin/python
Python 3.12.13
FastQC v0.12.1
samtools 1.23
2.7.11b
/home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts
